# Well Bundle

## 1. Importing / Installing Packages

In [1]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass, field
from typing import Dict, Iterable, List, Mapping, MutableMapping, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd

import os
import glob

# Optional geospatial stack (recommended)
try:
    from shapely.geometry import Point, MultiPoint
    from shapely.geometry.base import BaseGeometry
    HAVE_SHAPELY = True
except Exception:  # pragma: no cover
    HAVE_SHAPELY = False
    BaseGeometry = object  # type: ignore

try:
    import geopandas as gpd
    HAVE_GPD = True
except Exception:  # pragma: no cover
    HAVE_GPD = False

try:
    from pyproj import Transformer
    HAVE_PYPROJ = True
except Exception:  # pragma: no cover
    HAVE_PYPROJ = False

# Optional Bayesian optimizer
try:
    import optuna  # pip install optuna
    HAVE_OPTUNA = True
except Exception:
    HAVE_OPTUNA = False
    optuna = None  # type: ignore

c:\Users\apoorva.saxena\Desktop\Python_Conda_Projects\Parent_Child_Spacing\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Defining Functions

### 2.1. Defining Functions that is used in creating bundles

In [2]:
@dataclass
class WellBundleDBSCAN:
    """
    DBSCAN-style bundling for horizontal wells using *spacing output* pairs.

    Instead of recomputing distances from coordinates, this class builds an ε-neighbor
    graph directly from your spacing output (pairwise rows like well_i, well_k, distances,
    angle, overlap, etc.) and then runs a classic DBSCAN region-growing algorithm.

    Core idea
    ---------
    - Build an undirected graph where an edge (i,k) exists if the *effective* pair distance
      is ≤ ε (in feet). Effective distance combines a horizontal spacing metric and an
      optional angle penalty:

          effective_dist_ft = base_horizontal_ft + mu_ft_per_rad * clip(|Δθ|, max=angle_clip_deg)_rad

    - Run DBSCAN: a well seeds/expands a cluster iff (neighbors + itself) ≥ min_samples.
      Border points attached to any cluster are included; isolated points become NOISE (-1).

    Typical column mapping from your spacing output
    -----------------------------------------------
    * Candidate base distances (first present is used automatically):
        - 'horizontal_crossline_mean_ft'   (best if present for parallel-like)
        - 'hz_effective'
        - 'horizontal_dist_effective'
        - 'horizontal_dist_median'
        - 'horizontal_dist'
        - 'mean_windowed_ft'
        - 'min_distance_ft'

    * Optional filters & features:
        - 'pair_alignment'  (e.g., 'parallel_like')  -> set require_parallel_like=True to keep only these
        - 'reject_reason'   (drop pairs with a value)
        - 'angle_deg'       (angular misalignment between laterals)

    Parameters
    ----------
    epsilon_ft : float
        DBSCAN neighborhood radius in feet (applied to `effective_dist_ft`).
        Start ~300–800 ft for parallel-lateral bundles.
    min_samples : int
        Minimum (neighbors + itself) to seed/expand a cluster (DBSCAN density).
    mu_ft_per_rad : float
        Angle penalty scale: feet added per radian of |Δθ|.
        Example: 600 ft/rad ⇒ ~314 ft added at 30°.
    angle_col : str
        Column name for angle misalignment in degrees. If missing in data, angle penalty is 0.
    angle_clip_deg : float
        Cap |Δθ| at this value to avoid huge penalties from outliers (default 60°).
    require_parallel_like : bool
        If True, keep only rows whose `pair_alignment` contains 'parallel'.
        Leave True for Option B (parallel-laterals); set False to ignore this filter.
    reject_col : str
        Column indicating rejected pairs; non-null rows are dropped.
    base_candidates : Sequence[str]
        Preference order for base horizontal distance column.

    Attributes (after fit)
    ----------------------
    labels_ : Dict[str, int]
        Mapping well -> cluster id (NOISE = -1). Well ids are strings.
    well_bundles_ : pd.DataFrame
        Two columns: ['well', 'bundle_id'].
    edges_ : pd.DataFrame
        Filtered edges used for clustering with ['well_i','well_k','effective_dist_ft', base_col, angle_col?].
    base_col_ : str
        The chosen base distance column actually used.

    Usage
    -----
    >>> # spacing_df: your spacing output pairs
    >>> model = WellBundleDBSCAN(epsilon_ft=600, min_samples=3, mu_ft_per_rad=600)
    >>> model.fit(spacing_df)
    >>> well_bundles = model.get_well_bundles()
    >>> bundle_stats = model.get_bundle_stats()
    >>>
    >>> # If you have representative well points (normalized midpoints) for hulls:
    >>> # points_df columns: 'well' + either ['x','y'] in a planar CRS (ft/m) OR ['lon','lat'] (EPSG:4326)
    >>> hulls = model.build_bundle_hulls(points_df, id_col="well", x_col="x", y_col="y")
    """
    epsilon_ft: float = 600.0
    min_samples: int = 3
    mu_ft_per_rad: float = 600.0
    angle_col: str = "angle_deg"
    angle_clip_deg: float = 60.0
    require_parallel_like: bool = True
    reject_col: str = "reject_reason"
    base_candidates: Sequence[str] = field(default_factory=lambda: (
        "horizontal_crossline_mean_ft",
        "hz_effective",
        "horizontal_dist_effective",
        "horizontal_dist_median",
        "horizontal_dist",
        "mean_windowed_ft",
        "min_distance_ft",
    ))

    # Fitted artifacts
    labels_: Dict[str, int] = field(init=False, default_factory=dict)
    well_bundles_: pd.DataFrame = field(init=False)        # ['well','bundle_id']
    edges_: pd.DataFrame = field(init=False)               # filtered edges with effective distances
    base_col_: str = field(init=False, default="")

    # --------------------------- public API ---------------------------

    def fit(self, spacing_df: pd.DataFrame) -> "WellBundleDBSCAN":
        """
        Build the ε-neighbor graph from spacing pairs and run DBSCAN.

        Parameters
        ----------
        spacing_df : pd.DataFrame
            Must contain 'well_i','well_k' and at least one of `base_candidates`.
            Optionally include `angle_col`, 'pair_alignment', and `reject_col`.

        Returns
        -------
        self
        """
        nbrs, edges, base_col = self._build_neighbors(spacing_df)
        self.base_col_ = base_col
        self.edges_ = edges
        self.labels_ = self._dbscan_from_neighbors(nbrs, self.min_samples)
        self.well_bundles_ = (
            pd.Series(self.labels_, name="bundle_id").rename_axis("well").reset_index()
        )
        self.well_bundles_["bundle_id"] = self.well_bundles_["bundle_id"].astype(int)
        return self

    def get_well_bundles(self) -> pd.DataFrame:
        """
        Returns
        -------
        pd.DataFrame
            Columns: ['well','bundle_id'] (NOISE = -1).
        """
        self._require_fitted()
        return self.well_bundles_.copy()

    def get_bundle_stats(self) -> pd.DataFrame:
        """
        Aggregate edge-level stats for wells that landed in the *same* bundle.

        Returns
        -------
        pd.DataFrame
            One row per bundle_id (excluding -1) with columns:
            ['bundle_id','n_wells','n_edges','mean_eff_ft','median_eff_ft','base_dist_mean','mean_angle_deg?']
        """
        self._require_fitted()
        e = self.edges_.copy()
        lab = self.well_bundles_.set_index("well")["bundle_id"]
        e["bundle_i"] = e["well_i"].map(lab)
        e["bundle_k"] = e["well_k"].map(lab)
        same = e["bundle_i"].eq(e["bundle_k"]) & (e["bundle_i"] != -1)

        agg_dict: Dict[str, Union[str, List[str]]] = {
            "effective_dist_ft": ["count", "mean", "median"],
            self.base_col_: "mean",
        }
        if self.angle_col in e.columns:
            agg_dict[self.angle_col] = "mean"

        g = e.loc[same].groupby("bundle_i").agg(agg_dict)
        cols = ["n_edges", "mean_eff_ft", "median_eff_ft", "base_dist_mean"]
        if self.angle_col in e.columns:
            cols.append("mean_angle_deg")
        g.columns = cols

        n_wells = (
            self.well_bundles_[self.well_bundles_["bundle_id"] != -1]
            .groupby("bundle_id")["well"].nunique()
        )
        out = g.join(n_wells.rename("n_wells")).reset_index().rename(columns={"bundle_i": "bundle_id"})
        # Order by size (desc), then tightness
        out = out.sort_values(["n_wells", "median_eff_ft"], ascending=[False, True], ignore_index=True)
        return out

    def build_bundle_hulls(
        self,
        well_points_df: pd.DataFrame,
        *,
        id_col: str = "well",
        x_col: Optional[str] = None,
        y_col: Optional[str] = None,
        lon_col: Optional[str] = None,
        lat_col: Optional[str] = None,
        input_crs_epsg: int = 4326,
        out_crs_epsg: int = 3857,
        return_geopandas: bool = True,
        min_points_for_bundle: int = 1,
    ) -> Union[pd.DataFrame, "gpd.GeoDataFrame"]:
        """
        Create convex hull geometries (bundle polygons) from representative well points.

        You may supply either planar X/Y *or* lon/lat. If lon/lat are provided, points
        are projected to EPSG:3857 before hulls are computed. Noise wells (bundle -1)
        are ignored.

        Parameters
        ----------
        well_points_df : pd.DataFrame
            Representative point per well (e.g., normalized midpoint). Must include `id_col`
            and either (x_col,y_col) or (lon_col,lat_col).
        id_col : str
            Column containing well ids that match `get_well_bundles()['well']`.
        x_col, y_col : Optional[str]
            Planar coordinates (ft or m) in a consistent CRS. If provided, used directly.
        lon_col, lat_col : Optional[str]
            Geographic coordinates in degrees (EPSG:4326). If provided, will be projected.
        input_crs_epsg : int
            EPSG of input lon/lat (default 4326).
        out_crs_epsg : int
            EPSG to project into for hull computation (default 3857).
        return_geopandas : bool
            If True and GeoPandas is available, returns a GeoDataFrame. Otherwise returns
            a DataFrame with a 'geometry_wkt' column.
        min_points_for_bundle : int
            Minimum number of wells in a bundle to emit a hull. Smaller bundles are dropped.

        Returns
        -------
        GeoDataFrame | DataFrame
            One row per bundle with columns:
            ['bundle_id','n_wells','centroid_x','centroid_y','area','perimeter','geometry']
            (or 'geometry_wkt' if GeoPandas/Shapely unavailable)

        Notes
        -----
        - Shapely/GeoPandas strongly recommended. Without them, geometry export is limited.
        - For bundles with 1–2 points, the convex hull is a Point/LineString (area = 0).
        """
        self._require_fitted()
        if not HAVE_SHAPELY:
            # Fallback: return centroids only (no true polygons)
            df = self._fallback_hulls_no_shapely(well_points_df, id_col=id_col)
            if return_geopandas and HAVE_GPD:
                return gpd.GeoDataFrame(df, geometry=None)
            return df

        # Attach bundle labels to points
        labels = self.get_well_bundles()
        pts = well_points_df.copy()
        pts[id_col] = pts[id_col].astype(str)
        lab = labels.set_index("well")["bundle_id"]
        pts["bundle_id"] = pts[id_col].map(lab)
        pts = pts[pts["bundle_id"].notna() & (pts["bundle_id"] != -1)]
        if pts.empty:
            return gpd.GeoDataFrame(columns=["bundle_id","n_wells","centroid_x","centroid_y","area","perimeter","geometry"]) if (return_geopandas and HAVE_GPD) else \
                   pd.DataFrame(columns=["bundle_id","n_wells","centroid_x","centroid_y","area","perimeter","geometry_wkt"])

        # Build planar X/Y
        if (x_col and y_col) and (x_col in pts.columns) and (y_col in pts.columns):
            X = pts[[x_col, y_col]].to_numpy(dtype=float, copy=False)
        elif (lon_col and lat_col) and (lon_col in pts.columns) and (lat_col in pts.columns):
            if not HAVE_PYPROJ:
                raise RuntimeError("pyproj is required to project lon/lat to planar coordinates for hulls.")
            transformer = Transformer.from_crs(f"EPSG:{input_crs_epsg}", f"EPSG:{out_crs_epsg}", always_xy=True)
            xx, yy = transformer.transform(pts[lon_col].to_numpy(float), pts[lat_col].to_numpy(float))
            X = np.column_stack([xx, yy])
        else:
            raise ValueError("Provide either (x_col,y_col) or (lon_col,lat_col) in well_points_df.")

        pts["_x"] = X[:, 0]
        pts["_y"] = X[:, 1]

        rows = []
        for bid, grp in pts.groupby("bundle_id", sort=False):
            if len(grp) < min_points_for_bundle:
                continue
            mpts = MultiPoint([Point(xy) for xy in grp[["_x","_y"]].to_numpy()])
            hull = mpts.convex_hull  # Point/LineString/Polygon
            n_wells = int(grp[id_col].nunique())
            centroid = hull.centroid
            area = float(getattr(hull, "area", 0.0))
            perimeter = float(getattr(hull, "length", 0.0))
            rows.append(
                dict(bundle_id=int(bid),
                     n_wells=n_wells,
                     centroid_x=float(centroid.x),
                     centroid_y=float(centroid.y),
                     area=area,
                     perimeter=perimeter,
                     geometry=hull)
            )

        hulls_df = pd.DataFrame(rows)
        if return_geopandas and HAVE_GPD:
            return gpd.GeoDataFrame(hulls_df, geometry="geometry", crs=f"EPSG:{out_crs_epsg}")
        else:
            # Provide WKT if GeoPandas not requested/available
            out = hulls_df.copy()
            out["geometry_wkt"] = out["geometry"].apply(lambda g: g.wkt if HAVE_SHAPELY else None)
            return out.drop(columns=["geometry"])

    # --------------------------- internals ---------------------------

    def _choose_base_col(self, df: pd.DataFrame) -> str:
        for c in self.base_candidates:
            if c in df.columns:
                return c
        raise KeyError(
            f"Could not find any base distance column from candidates: {self.base_candidates}"
        )

    def _build_neighbors(self, spacing_df: pd.DataFrame) -> Tuple[Dict[str, List[str]], pd.DataFrame, str]:
        df = spacing_df.copy()

        # base distance
        base_col = self._choose_base_col(df)
        base = pd.to_numeric(df[base_col], errors="coerce")

        # angle penalty (if available)
        if self.angle_col in df.columns:
            ang_deg = pd.to_numeric(df[self.angle_col], errors="coerce").abs()
            ang_deg = ang_deg.clip(upper=self.angle_clip_deg).fillna(self.angle_clip_deg)
            ang_rad = np.deg2rad(ang_deg)
            angle_penalty = self.mu_ft_per_rad * ang_rad
        else:
            angle_penalty = 0.0

        # filters
        mask = np.isfinite(base)
        if self.require_parallel_like and ("pair_alignment" in df.columns):
            mask &= df["pair_alignment"].str.contains("parallel", case=False, na=False)
        if self.reject_col in df.columns:
            rej = df[self.reject_col]
            # keep rows where reject_reason is NaN OR empty/whitespace
            ok_reject = rej.isna() | (rej.astype(str).str.strip() == "")
            mask &= ok_reject


        eff = base + angle_penalty
        filt = df.loc[mask].copy()
        filt["effective_dist_ft"] = eff[mask]

        # build ε-edges
        nbrs = filt.loc[filt["effective_dist_ft"] <= self.epsilon_ft, ["well_i", "well_k"]].dropna()
        nbrs["well_i"] = nbrs["well_i"].astype(str)
        nbrs["well_k"] = nbrs["well_k"].astype(str)

        undirected = pd.concat(
            [nbrs, nbrs.rename(columns={"well_i": "well_k", "well_k": "well_i"})],
            ignore_index=True
        ).drop_duplicates()

        neighbor_dict: Dict[str, List[str]] = (
            undirected.groupby("well_i")["well_k"].apply(lambda s: list(pd.unique(s))).to_dict()
        )

        # Ensure all wells appear
        all_wells = pd.unique(pd.concat([df["well_i"], df["well_k"]], ignore_index=True).dropna()).astype(str)
        for w in all_wells:
            neighbor_dict.setdefault(w, [])

        keep_cols = ["well_i", "well_k", "effective_dist_ft", base_col]
        if self.angle_col in df.columns:
            keep_cols.append(self.angle_col)
        edges = filt[keep_cols].copy()
        edges["well_i"] = edges["well_i"].astype(str)
        edges["well_k"] = edges["well_k"].astype(str)
        return neighbor_dict, edges, base_col

    @staticmethod
    def _dbscan_from_neighbors(neighbor_dict: Mapping[str, List[str]], min_samples: int) -> Dict[str, int]:
        UNVISITED = 0
        NOISE = -1
        labels: Dict[str, int] = {node: UNVISITED for node in neighbor_dict.keys()}
        cid = 0

        for node in list(labels.keys()):
            if labels[node] != UNVISITED:
                continue
            neighbors = neighbor_dict.get(node, [])
            if (len(neighbors) + 1) < min_samples:  # +1 counts itself
                labels[node] = NOISE
                continue
            cid += 1
            labels[node] = cid
            queue: deque[str] = deque(neighbors)
            while queue:
                n = queue.popleft()
                if labels.get(n, UNVISITED) == NOISE:
                    labels[n] = cid
                if labels.get(n, UNVISITED) != UNVISITED:
                    continue
                labels[n] = cid
                n_neighbors = neighbor_dict.get(n, [])
                if (len(n_neighbors) + 1) >= min_samples:
                    queue.extend(n_neighbors)
        return labels

    def _require_fitted(self) -> None:
        if not hasattr(self, "well_bundles_") or self.well_bundles_ is None:
            raise RuntimeError("Call `.fit(spacing_df)` first.")

    def _fallback_hulls_no_shapely(self, well_points_df: pd.DataFrame, *, id_col: str = "well") -> pd.DataFrame:
        labels = self.get_well_bundles()
        pts = well_points_df.copy()
        pts[id_col] = pts[id_col].astype(str)
        lab = labels.set_index("well")["bundle_id"]
        pts["bundle_id"] = pts[id_col].map(lab)
        pts = pts[pts["bundle_id"].notna() & (pts["bundle_id"] != -1)]
        if pts.empty:
            return pd.DataFrame(columns=["bundle_id","n_wells","centroid_x","centroid_y","area","perimeter","geometry_wkt"])
        # crude centroid only (no real geometry without shapely)
        cent = pts.groupby("bundle_id")[["x","y"]].mean().rename(columns={"x":"centroid_x","y":"centroid_y"})
        n = pts.groupby("bundle_id")[id_col].nunique().rename("n_wells")
        out = pd.concat([n, cent], axis=1).reset_index()
        out["area"] = 0.0
        out["perimeter"] = 0.0
        out["geometry_wkt"] = None
        return out

In [3]:
def _score_bundles(
    bundle_stats: pd.DataFrame,
    well_bundles: pd.DataFrame,
    epsilon_ft: float,
    weights: Tuple[float, float, float] = (2.0, 1.0, 0.5),
) -> float:
    """
    Compute a single scalar score to MAXIMIZE.

    Components
    ----------
    coverage:         fraction of wells assigned to clusters (higher is better)
    normalized_tight: weighted mean of (median_eff_ft / epsilon_ft), weight = n_wells (lower is better)
    max_bundle_share: largest bundle share among assigned wells (lower is better)

    score = + w1 * coverage - w2 * normalized_tight - w3 * max_bundle_share
    """
    w1, w2, w3 = weights
    if well_bundles.empty:
        return -1e9

    total_wells = int(well_bundles["well"].nunique())
    assigned = well_bundles.query("bundle_id != -1")
    n_assigned = int(assigned["well"].nunique())
    if n_assigned == 0 or epsilon_ft <= 0:
        return -1e9

    coverage = n_assigned / max(total_wells, 1)

    if bundle_stats.empty:
        # if we have no within-bundle edges, deem it low quality
        return w1 * coverage - w2 * 1.0 - w3 * 1.0

    # normalized tightness (lower better), weighted by n_wells
    tight = (bundle_stats["median_eff_ft"] / epsilon_ft).to_numpy()
    weights_w = np.maximum(bundle_stats["n_wells"].to_numpy(), 1)
    normalized_tight = float(np.average(tight, weights=weights_w))

    # dominance penalty: avoid one mega-cluster
    sizes = bundle_stats["n_wells"].to_numpy()
    max_bundle_share = float(np.max(sizes) / max(n_assigned, 1))

    score = + w1 * coverage - w2 * normalized_tight - w3 * max_bundle_share
    return float(score)


def evaluate_params_once(
    spacing_df: pd.DataFrame,
    *,
    epsilon_ft: float,
    min_samples: int,
    mu_ft_per_rad: float,
    require_parallel_like: bool = True,
    angle_col: str = "angle_deg",
    angle_clip_deg: float = 60.0,
    scoring_weights: Tuple[float, float, float] = (2.0, 1.0, 0.5),
) -> Tuple[float, "WellBundleDBSCAN", pd.DataFrame, pd.DataFrame]:
    """
    Fit a WellBundleDBSCAN model for given params and return (score, model, well_bundles, bundle_stats).
    """
    model = WellBundleDBSCAN(
        epsilon_ft=float(epsilon_ft),
        min_samples=int(min_samples),
        mu_ft_per_rad=float(mu_ft_per_rad),
        angle_col=angle_col,
        angle_clip_deg=float(angle_clip_deg),
        require_parallel_like=bool(require_parallel_like),
    )
    model.fit(spacing_df)

    well_bundles = model.get_well_bundles()
    bundle_stats = model.get_bundle_stats()
    score = _score_bundles(bundle_stats, well_bundles, epsilon_ft=float(epsilon_ft), weights=scoring_weights)
    return score, model, well_bundles, bundle_stats


def optimize_bundling_params(
    spacing_df: pd.DataFrame,
    *,
    # Search spaces (edit to your basin)
    epsilon_bounds_ft: Tuple[float, float] = (300.0, 900.0),
    min_samples_bounds: Tuple[int, int] = (3, 6),
    mu_bounds_ft_per_rad: Tuple[float, float] = (300.0, 1200.0),
    require_parallel_like: bool = True,   # Option B default
    # Scoring weights
    scoring_weights: Tuple[float, float, float] = (2.0, 1.0, 0.5),
    # Optimizer controls
    n_trials: int = 40,
    random_seed: int = 42,
    show_progress: bool = True,
) -> Dict[str, object]:
    """
    Black-box parameter optimizer for WellBundleDBSCAN using Bayesian/TPE if available.

    Objective (maximize)
    --------------------
    score = + 2.0 * coverage
            - 1.0 * normalized_tightness (median_eff_ft / epsilon_ft; weighted by bundle size)
            - 0.5 * max_bundle_share

    Returns
    -------
    dict with keys:
        'best_params'   : dict(epsilon_ft, min_samples, mu_ft_per_rad)
        'best_score'    : float
        'best_model'    : fitted WellBundleDBSCAN
        'well_bundles'  : DataFrame of well -> bundle_id
        'bundle_stats'  : DataFrame of bundle stats
        'study'         : Optuna study or None
        'trials'        : DataFrame of tried params/scores
    """
    rng = np.random.default_rng(seed=random_seed)

    # sanitize bounds
    eps_lo, eps_hi = map(float, epsilon_bounds_ft)
    mu_lo, mu_hi = map(float, mu_bounds_ft_per_rad)
    ms_lo, ms_hi = map(int, min_samples_bounds)
    ms_lo = max(2, ms_lo)
    ms_hi = max(ms_lo, ms_hi)

    trial_rows = []
    best = {
        "best_params": None,
        "best_score": -1e18,
        "best_model": None,
        "well_bundles": pd.DataFrame(),
        "bundle_stats": pd.DataFrame(),
        "study": None,
        "trials": pd.DataFrame(),
    }

    if HAVE_OPTUNA:
        sampler = optuna.samplers.TPESampler(seed=random_seed)
        study = optuna.create_study(direction="maximize", sampler=sampler)

        def _objective(trial: "optuna.trial.Trial") -> float:
            epsilon_ft = trial.suggest_float("epsilon_ft", eps_lo, eps_hi)
            min_samples = trial.suggest_int("min_samples", ms_lo, ms_hi)
            mu_ft_per_rad = trial.suggest_float("mu_ft_per_rad", mu_lo, mu_hi)

            score, model, wb, bs = evaluate_params_once(
                spacing_df,
                epsilon_ft=epsilon_ft,
                min_samples=min_samples,
                mu_ft_per_rad=mu_ft_per_rad,
                require_parallel_like=require_parallel_like,
                scoring_weights=scoring_weights,
            )
            trial_rows.append({
                "epsilon_ft": epsilon_ft,
                "min_samples": min_samples,
                "mu_ft_per_rad": mu_ft_per_rad,
                "score": score,
                "n_clusters": int((wb["bundle_id"] != -1).sum() - (wb["bundle_id"] != -1).sum() + bs.shape[0]) if not bs.empty else 0,
                "coverage": float((wb["bundle_id"] != -1).mean()) if not wb.empty else 0.0,
            })

            # track best
            if score > best["best_score"]:
                best.update({
                    "best_params": {"epsilon_ft": epsilon_ft, "min_samples": min_samples, "mu_ft_per_rad": mu_ft_per_rad},
                    "best_score": score,
                    "best_model": model,
                    "well_bundles": wb,
                    "bundle_stats": bs,
                })
            return score

        study.optimize(_objective, n_trials=n_trials, show_progress_bar=show_progress)
        best["study"] = study
        best["trials"] = pd.DataFrame(trial_rows)
        return best

    # ---- Fallback: random search (no Optuna) ----
    n_rand = max(25, n_trials)
    for _ in range(n_rand):
        epsilon_ft = float(rng.uniform(eps_lo, eps_hi))
        min_samples = int(rng.integers(ms_lo, ms_hi + 1))
        mu_ft_per_rad = float(rng.uniform(mu_lo, mu_hi))

        score, model, wb, bs = evaluate_params_once(
            spacing_df,
            epsilon_ft=epsilon_ft,
            min_samples=min_samples,
            mu_ft_per_rad=mu_ft_per_rad,
            require_parallel_like=require_parallel_like,
            scoring_weights=scoring_weights,
        )
        trial_rows.append({
            "epsilon_ft": epsilon_ft,
            "min_samples": min_samples,
            "mu_ft_per_rad": mu_ft_per_rad,
            "score": score,
            "n_clusters": int((wb["bundle_id"] != -1).sum() - (wb["bundle_id"] != -1).sum() + bs.shape[0]) if not bs.empty else 0,
            "coverage": float((wb["bundle_id"] != -1).mean()) if not wb.empty else 0.0,
        })

        if score > best["best_score"]:
            best.update({
                "best_params": {"epsilon_ft": epsilon_ft, "min_samples": min_samples, "mu_ft_per_rad": mu_ft_per_rad},
                "best_score": score,
                "best_model": model,
                "well_bundles": wb,
                "bundle_stats": bs,
            })

    best["trials"] = pd.DataFrame(trial_rows)
    return best


### 2.1. Defining Functions that is used to read parquet files from a given folder to pandas dataframe

In [4]:
def read_parquet_folder_to_dataframe(folder_path):
    """
    Reads all Parquet files from a specified folder and concatenates them
    into a single pandas DataFrame.

    Args:
        folder_path (str): The path to the folder containing the Parquet files.

    Returns:
        pandas.DataFrame: A DataFrame containing the combined data from all
                          Parquet files in the folder.
                          Returns an empty DataFrame if no Parquet files are found.
    """
    # Construct the pattern to find all .parquet files in the folder
    parquet_files_pattern = os.path.join(folder_path, '*.parquet')
    
    # Get a list of all Parquet file paths in the specified folder
    all_parquet_files = glob.glob(parquet_files_pattern)

    if not all_parquet_files:
        print(f"No Parquet files found in the folder: {folder_path}")
        return pd.DataFrame() # Return an empty DataFrame if no files are found

    # Read each Parquet file into a DataFrame and store them in a list
    dataframes = [pd.read_parquet(file_path) for file_path in all_parquet_files]

    # Concatenate all DataFrames in the list into a single DataFrame
    combined_df = pd.concat(dataframes, ignore_index=True)

    return combined_df

# Example usage:
# Assuming you have a folder named 'my_parquet_data' with some .parquet files
# combined_data = read_parquet_folder_to_dataframe('my_parquet_data')
# print(combined_data.head())

## 2. Testing

In [5]:
# Readind the parquet file from the specified folder path
folder_path = r"C:\Users\apoorva.saxena\Desktop\Python_Conda_Projects\Parent_Child_Spacing\notebooks\SpacingStats"

df_spacing = read_parquet_folder_to_dataframe(folder_path)

# Filter out rows where 'reject_reason' is not empty
df_spacing_filt = df_spacing[df_spacing["reject_reason"]==""].reset_index(drop=True).copy()

In [6]:
df_spacing_filt

,well_i,well_k,horizontal_dist,horizontal_dist_median,vertical_dist,3D_dist,drill_direction_i,drill_direction_k,n_samples,dy_p5,...,contact_len_i_interior_ft,contact_pct_i_interior,contact_len_i_ft_T300,contact_pct_i_T300,contact_len_i_interior_ft_T300,contact_pct_i_interior_T300,horizontal_crossline_mean_ft,hz_effective,hz_basis,3D_dist_effective
0,30025410040100,30025455300000,2834.430974,2836.702875,29.4475,2834.583938,NS,NS,41.0,2809.687783,...,NaN,NaN,NaN,NaN,NaN,NaN,2834.430974,2834.430974,crossline_mean,2834.583938
1,30025410040100,30025503690000,933.507723,940.956180,23.6125,933.806307,NS,NS,42.0,900.076464,...,NaN,NaN,NaN,NaN,NaN,NaN,933.507723,933.507723,crossline_mean,933.806307
2,30025410040100,42501364850000,2870.944865,2849.950825,4.1775,2870.947904,NS,NS,34.0,2741.556240,...,NaN,NaN,NaN,NaN,NaN,NaN,2870.944865,2870.944865,crossline_mean,2870.947904
3,30025410040100,42501372540000,1505.026059,1505.095305,27.3745,1505.274992,NS,NS,3.0,1500.096522,...,NaN,NaN,NaN,NaN,NaN,NaN,1505.026059,1505.026059,crossline_mean,1505.274992
4,30025410040100,42501373440000,2918.138197,2920.880702,29.2850,2918.285139,NS,NS,4.0,2830.869039,...,NaN,NaN,NaN,NaN,NaN,NaN,2918.138197,2918.138197,crossline_mean,2918.285139
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15743,42501376110000,42501373040000,668.991547,677.768394,61.9835,671.856863,NS,NS,50.0,557.523843,...,NaN,NaN,NaN,NaN,NaN,NaN,668.991547,668.991547,crossline_mean,671.856863
15744,42501376110000,42501373050000,834.861571,823.656159,46.1885,836.138278,NS,NS,51.0,722.752425,...,NaN,NaN,NaN,NaN,NaN,NaN,834.861571,834.861571,crossline_mean,836.138278
15745,42501376110000,42501375350000,2093.235707,2095.556100,1.7250,2093.236418,NS,NS,50.0,2039.384051,...,NaN,NaN,NaN,NaN,NaN,NaN,2093.235707,2093.235707,crossline_mean,2093.236418
15746,42501376110000,42501375360000,2761.563591,2766.038907,25.3250,2761.679711,NS,NS,52.0,2710.968444,...,NaN,NaN,NaN,NaN,NaN,NaN,2761.563591,2761.563591,crossline_mean,2761.679711


In [14]:
# spacing_df is your spacing output dataframe

best = optimize_bundling_params(
    df_spacing,
    epsilon_bounds_ft=(1200, 2500),
    min_samples_bounds=(3,6),
    mu_bounds_ft_per_rad=(300, 900),
    require_parallel_like=True,          # Option B (parallel-laterals)
    scoring_weights=(2.0, 1.0, 0.5),     # (coverage, tightness, dominance)
    n_trials=40,                         # increase for more thorough search
    random_seed=1337,
)

print("Best params:", best["best_params"])
print("Best score:", best["best_score"])


well_bundles = best["well_bundles"]
bundle_stats = best["bundle_stats"]

# If you want to use the fitted model:
model = best["best_model"]
# ...and optionally build hulls later with your midpoints table:
# hulls = model.build_bundle_hulls(midpoints_df, id_col="well", x_col="x", y_col="y")


[I 2025-11-08 00:09:20,402] A new study created in memory with name: no-name-8e749af3-aeb1-4cd8-a37d-595ea296fcc6
Best trial: 0. Best value: 0.379528:   2%|▎         | 1/40 [00:00<00:13,  2.98it/s]

[I 2025-11-08 00:09:20,730] Trial 0 finished with value: 0.3795275058755019 and parameters: {'epsilon_ft': 1540.6320775202562, 'min_samples': 3, 'mu_ft_per_rad': 466.87591169661584}. Best is trial 0 with value: 0.3795275058755019.


Best trial: 0. Best value: 0.379528:   5%|▌         | 2/40 [00:00<00:12,  3.06it/s]

[I 2025-11-08 00:09:21,050] Trial 1 finished with value: 0.33084404933791545 and parameters: {'epsilon_ft': 1797.1119533789365, 'min_samples': 4, 'mu_ft_per_rad': 611.0356923585223}. Best is trial 0 with value: 0.3795275058755019.


Best trial: 0. Best value: 0.379528:   8%|▊         | 3/40 [00:00<00:11,  3.15it/s]

...

[I 2025-11-08 00:09:34,368] Trial 37 finished with value: 0.5936075992309104 and parameters: {'epsilon_ft': 2317.421727460917, 'min_samples': 4, 'mu_ft_per_rad': 619.6772347402782}. Best is trial 13 with value: 0.8997841835306029.


Best trial: 13. Best value: 0.899784:  98%|█████████▊| 39/40 [00:14<00:00,  2.75it/s]

[I 2025-11-08 00:09:34,714] Trial 38 finished with value: 0.8586368205125336 and parameters: {'epsilon_ft': 2403.537509716569, 'min_samples': 3, 'mu_ft_per_rad': 656.2774207590022}. Best is trial 13 with value: 0.8997841835306029.


Best trial: 13. Best value: 0.899784: 100%|██████████| 40/40 [00:14<00:00,  2.73it/s]

[I 2025-11-08 00:09:35,047] Trial 39 finished with value: 0.3530955175799737 and parameters: {'epsilon_ft': 2498.6206458113315, 'min_samples': 6, 'mu_ft_per_rad': 338.0716446242618}. Best is trial 13 with value: 0.8997841835306029.
Best params: {'epsilon_ft': 2497.524362238699, 'min_samples': 3, 'mu_ft_per_rad': 305.33855616934756}
Best score: 0.8997841835306029


In [15]:
well_bundles

,well,bundle_id
0,30025410040100,1
1,30025421210000,1
2,30025426220000,1
3,30025428730000,1
4,30025437530100,1
...,...,...
1986,42501373500000,-1
1987,42501373740000,-1
1988,42501375340000,-1
1989,42501375950000,-1


In [16]:
bundle_stats

,bundle_id,n_edges,mean_eff_ft,median_eff_ft,base_dist_mean,mean_angle_deg,n_wells
0,1,3310,1762.741849,1775.715310,1755.624282,1.335588,487
1,34,6988,1518.318377,1469.992266,1505.076290,2.484834,265
2,11,709,1736.122599,1688.664016,1721.842033,2.679701,116
3,13,414,1745.602405,1747.718434,1740.241539,1.005949,80
4,22,327,1696.721213,1670.485424,1693.059731,0.687065,54
...,...,...,...,...,...,...,...
97,87,6,1756.024481,2097.105451,1753.652602,0.445075,3
98,89,4,2147.837713,2147.838020,2146.783972,0.197731,3
99,96,4,2150.148365,2150.246532,2147.552533,0.487099,3
100,65,4,2260.884046,2260.889451,2259.531058,0.253884,3


In [10]:
# Which base distance is being used?
tmp_model = WellBundleDBSCAN(require_parallel_like=True)
_ , _ , base_col = tmp_model._build_neighbors(df_spacing)  # uses your patched code
print("Base column:", base_col)

# Distribution of the base distances
b = pd.to_numeric(df_spacing[base_col], errors="coerce")
print(b.describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]))

# How many pairs survive the filters?
pair_alignment_ok = df_spacing.get("pair_alignment", "").astype(str).str.contains("parallel", case=False, na=False)
rej = df_spacing.get("reject_reason")
ok_reject = (rej.isna() | (rej.astype(str).str.strip() == "")) if "reject_reason" in df_spacing.columns else True
survivors = df_spacing[pair_alignment_ok & ok_reject]
print("Pairs after filters:", len(survivors))

# What fraction would be neighbors for a few ε values (ignoring angle penalty)?
for eps in [600, 900, 1200, 1800, 2400]:
    frac = np.mean(pd.to_numeric(survivors[base_col], errors="coerce") <= eps)
    print(f"<= {eps:4.0f} ft :", round(frac, 4))

# Same but with angle penalty (μ=600 ft/rad, clip 60°)
mu = 600.0
ang_deg = pd.to_numeric(survivors.get("angle_deg", 0), errors="coerce").abs().clip(upper=60).fillna(0.0)
eff = pd.to_numeric(survivors[base_col], errors="coerce") + mu * np.deg2rad(ang_deg)
for eps in [600, 900, 1200, 1800, 2400]:
    frac = np.mean(eff <= eps)
    print(f"(eff) <= {eps:4.0f} ft :", round(frac, 4))


Base column: horizontal_crossline_mean_ft
count    15748.000000
mean      1719.337330
std       1007.589023
min          0.824252
1%          25.754681
5%         249.032072
10%        467.650785
25%        935.123037
50%       1661.985309
75%       2454.639479
90%       2872.229121
95%       3083.165465
99%       4930.245615
max       8243.249595
Name: horizontal_crossline_mean_ft, dtype: float64
Pairs after filters: 14828
<=  600 ft : 0.1397
<=  900 ft : 0.2526
<= 1200 ft : 0.3578
<= 1800 ft : 0.5758
<= 2400 ft : 0.7646
(eff) <=  600 ft : 0.1334
(eff) <=  900 ft : 0.2465
(eff) <= 1200 ft : 0.3516
(eff) <= 1800 ft : 0.5684
(eff) <= 2400 ft : 0.7572
